# Day 3：从零手写 GPT 模型（本周最重要！）

> **目标**：从零实现完整的 GPT 模型，逐模块手写 `CausalSelfAttention → GPTBlock → GPT`，并验证与 HuggingFace GPT-2 权重的一致性。

---

## 实现路线图

```
Part 1: 基础组件
  NewGELU → LayerNorm（理解即可，用 nn.LayerNorm）

Part 2: CausalSelfAttention
  线性投影 → 分头 → QK^T/√d_k → 因果掩码 → softmax → V加权 → 合并 → 输出投影

Part 3: GPTBlock
  Pre-LN + CausalSelfAttention + Residual + Pre-LN + FFN + Residual

Part 4: 完整 GPT 模型
  Token Emb + Pos Emb → N × GPTBlock → Final LN → LM Head (权重共享)

Part 5: 验证
  加载 HuggingFace GPT-2 权重 → 对比输出 → 文本生成
```

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Part 1：模型配置

用一个 dataclass 管理所有超参数。这里默认使用 GPT-2 Small 的配置。

In [ ]:
from dataclasses import dataclass


@dataclass
class GPTConfig:
    """GPT 模型配置。默认值对应 GPT-2 Small (124M)。"""
    vocab_size: int = 50257
    block_size: int = 1024   # 最大序列长度（上下文窗口）
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True        # GPT-2 使用 bias，LLaMA 不使用


config = GPTConfig()
print(f"Config: {config}")
print(f"d_head = n_embd / n_head = {config.n_embd // config.n_head}")
print(f"d_ff = 4 * n_embd = {4 * config.n_embd}")

## Part 2：CausalSelfAttention — 因果多头自注意力

这是 GPT 最核心的组件。数学回顾：

$$
Q = xW_Q, \quad K = xW_K, \quad V = xW_V
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + \text{CausalMask}\right) V
$$

### 实现要点

1. 将 $W_Q, W_K, W_V$ 合并为一个 `(3 * n_embd, n_embd)` 的矩阵，一次投影后拆分——更高效
2. 因果掩码注册为 buffer（不参与梯度计算）
3. 分头时 reshape 为 `(B, n_head, T, d_head)`

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    因果多头自注意力。
    
    将 Q、K、V 的投影合并为一次线性变换，然后拆分为多头。
    使用下三角因果掩码，确保每个位置只能 attend 到 <= 自身位置的 token。
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0, "n_embd must be divisible by n_head"

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.d_head = config.n_embd // config.n_head

        # Q, K, V 合并投影: (n_embd) → (3 * n_embd)
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # 输出投影: (n_embd) → (n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # 因果掩码：下三角矩阵，注册为 buffer（不是参数，不参与梯度更新）
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()  # batch, seq_len, n_embd

        # Step 1: 一次投影得到 Q, K, V
        qkv = self.c_attn(x)               # (B, T, 3C)
        q, k, v = qkv.split(self.n_embd, dim=2)  # 各 (B, T, C)

        # Step 2: 分头 — reshape 为 (B, n_head, T, d_head)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)  # (B, nh, T, dk)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)  # (B, nh, T, dk)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)  # (B, nh, T, dk)

        # Step 3: 注意力分数 = QK^T / √d_k
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_head))  # (B, nh, T, T)

        # Step 4: 因果掩码 — 未来位置设为 -inf
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))

        # Step 5: Softmax → 注意力权重
        att = F.softmax(att, dim=-1)        # (B, nh, T, T)
        att = self.attn_dropout(att)

        # Step 6: 加权求和 V
        y = att @ v                          # (B, nh, T, dk)

        # Step 7: 合并多头 → 输出投影
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        y = self.resid_dropout(self.c_proj(y))

        return y

### 验证 CausalSelfAttention 的形状

In [ ]:
# 用小配置快速验证
test_config = GPTConfig(n_embd=64, n_head=4, block_size=32)
attn = CausalSelfAttention(test_config)

x = torch.randn(2, 10, 64)  # batch=2, seq_len=10, n_embd=64
y = attn(x)
print(f"Input shape:  {x.shape}")   # (2, 10, 64)
print(f"Output shape: {y.shape}")   # (2, 10, 64) — 形状不变！
assert y.shape == x.shape, "Attention 的输入输出形状应该一致"

### 可视化因果掩码

直观理解因果掩码如何阻止 "偷看未来":

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

T = 8
causal_mask = torch.tril(torch.ones(T, T))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图：因果掩码（0/1）
ax = axes[0]
im = ax.imshow(causal_mask.numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_title('Causal Mask (1=attend, 0=block)', fontsize=12)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
for i in range(T):
    for j in range(T):
        ax.text(j, i, f"{int(causal_mask[i, j])}", ha='center', va='center', fontsize=10)

# 右图：softmax 后的注意力（用随机分数模拟）
random_scores = torch.randn(T, T)
masked_scores = random_scores.masked_fill(causal_mask == 0, float('-inf'))
attn_weights = F.softmax(masked_scores, dim=-1)

ax = axes[1]
im = ax.imshow(attn_weights.detach().numpy(), cmap='Reds', vmin=0)
ax.set_title('Attention Weights (after softmax)', fontsize=12)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')

plt.tight_layout()
plt.show()
print("观察：每一行只有 <= 对角线位置有非零权重，未来位置的权重为 0")

## Part 3：FFN（前馈网络）

GPT-2 使用标准的两层 MLP，激活函数为 GELU：

$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$$

其中 $W_1 \in \mathbb{R}^{d \times 4d}$，$W_2 \in \mathbb{R}^{4d \times d}$。

In [ ]:
class MLP(nn.Module):
    """GPT-2 风格的前馈网络：Linear → GELU → Linear → Dropout。"""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU(approximate='tanh')  # GPT-2 使用 tanh 近似的 GELU
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c_fc(x)       # (B, T, C) → (B, T, 4C)
        x = self.gelu(x)       # (B, T, 4C)
        x = self.c_proj(x)     # (B, T, 4C) → (B, T, C)
        x = self.dropout(x)
        return x

## Part 4：GPTBlock — 一个完整的 Transformer Decoder 层

使用 **Pre-Norm** 结构：

$$h' = h + \text{CausalMHA}(\text{LN}(h))$$
$$h_{\text{out}} = h' + \text{FFN}(\text{LN}(h'))$$

In [ ]:
class GPTBlock(nn.Module):
    """一个 GPT Block = Pre-LN + Causal Attention + Residual + Pre-LN + FFN + Residual。"""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-Norm: LN 在子层之前
        x = x + self.attn(self.ln_1(x))  # 残差连接 + 注意力
        x = x + self.mlp(self.ln_2(x))   # 残差连接 + FFN
        return x

### 验证 GPTBlock

In [ ]:
test_config = GPTConfig(n_embd=64, n_head=4, block_size=32)
block = GPTBlock(test_config)

x = torch.randn(2, 10, 64)
y = block(x)
print(f"Input shape:  {x.shape}")   # (2, 10, 64)
print(f"Output shape: {y.shape}")   # (2, 10, 64)
assert y.shape == x.shape

n_params = sum(p.numel() for p in block.parameters())
print(f"Block parameters: {n_params:,}")
print(f"理论值 ≈ 12 * d^2 = {12 * 64**2:,} (忽略 bias 和 LN)")

## Part 5：完整的 GPT 模型

将所有组件组装起来：

```
Token Embedding + Position Embedding + Dropout
    → N × GPTBlock
    → Final LayerNorm
    → LM Head（与 Token Embedding 权重共享）
```

In [ ]:
class GPT(nn.Module):
    """完整的 GPT 语言模型。"""

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),   # token embedding
            wpe  = nn.Embedding(config.block_size, config.n_embd),   # position embedding
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([GPTBlock(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),                     # final layer norm
        ))

        # LM Head: 映射回词表空间
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # 权重共享: LM Head 与 Token Embedding 共享权重
        self.transformer.wte.weight = self.lm_head.weight

        # 初始化权重
        self.apply(self._init_weights)
        # GPT-2 残差投影层特殊初始化: 1/√(2N)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        n_params = sum(p.numel() for p in self.parameters())
        # 权重共享后 lm_head.weight 不算额外参数
        print(f"Model parameters: {n_params:,} ({n_params/1e6:.1f}M)")

    def _init_weights(self, module):
        """GPT-2 风格的权重初始化。"""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,
        targets: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        Args:
            idx: token indices, shape (B, T)
            targets: target token indices, shape (B, T)，训练时提供
        Returns:
            logits: (B, T, vocab_size)
            loss: scalar，仅在 targets 不为 None 时返回
        """
        B, T = idx.size()
        assert T <= self.config.block_size, f"序列长度 {T} 超过 block_size {self.config.block_size}"

        # Token + Position Embedding
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)  # (T,)
        tok_emb = self.transformer.wte(idx)    # (B, T, C)
        pos_emb = self.transformer.wpe(pos)    # (T, C) → broadcast to (B, T, C)
        x = self.transformer.drop(tok_emb + pos_emb)

        # N 层 GPTBlock
        for block in self.transformer.h:
            x = block(x)

        # Final LayerNorm + LM Head
        x = self.transformer.ln_f(x)           # (B, T, C)
        logits = self.lm_head(x)               # (B, T, V)

        # 计算 loss
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),  # (B*T, V)
                targets.view(-1),                   # (B*T,)
            )

        return logits, loss

### 验证完整模型

In [ ]:
# 用小配置验证
small_config = GPTConfig(
    vocab_size=100,
    block_size=32,
    n_layer=4,
    n_head=4,
    n_embd=64,
    dropout=0.0,
)
model = GPT(small_config)

# 前向传播测试
idx = torch.randint(0, 100, (2, 16))       # (batch=2, seq_len=16)
targets = torch.randint(0, 100, (2, 16))
logits, loss = model(idx, targets)

print(f"\nInput shape:   {idx.shape}")
print(f"Logits shape:  {logits.shape}")    # (2, 16, 100)
print(f"Loss:          {loss.item():.4f}")
print(f"Expected loss ≈ -ln(1/100) = {-math.log(1/100):.4f} (随机初始化)")

## Part 6：加载 HuggingFace GPT-2 预训练权重

验证我们手写模型的正确性：加载官方 GPT-2 权重，对比输出。

HuggingFace 的参数命名与我们稍有不同，需要做映射。

In [ ]:
@classmethod
def from_pretrained(cls, model_type: str = 'gpt2') -> 'GPT':
    """
    从 HuggingFace 加载预训练 GPT-2 权重。
    model_type: 'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'
    """
    from transformers import GPT2LMHeadModel

    config_args = {
        'gpt2':        dict(n_layer=12, n_head=12, n_embd=768),
        'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),
        'gpt2-large':  dict(n_layer=36, n_head=20, n_embd=1280),
        'gpt2-xl':     dict(n_layer=48, n_head=25, n_embd=1600),
    }[model_type]

    config = GPTConfig(
        block_size=1024,
        vocab_size=50257,
        dropout=0.0,  # 推理时不需要 dropout
        **config_args,
    )

    # 初始化我们的模型
    model = cls(config)
    sd = model.state_dict()

    # 加载 HuggingFace 模型
    print(f"Loading HuggingFace {model_type}...")
    model_hf = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()

    # HF 的 attn.c_attn 和 attn.c_proj 使用 Conv1D，需要转置
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight',
                  'mlp.c_fc.weight', 'mlp.c_proj.weight']

    for k in sd_hf:
        if k.endswith('.attn.masked_bias') or k.endswith('.attn.bias'):
            continue  # 跳过 HF 的 causal mask buffer

        # 将 HF 的参数名映射到我们的模型
        target_key = k
        if any(k.endswith(w) for w in transposed):
            assert sd_hf[k].shape[::-1] == sd[target_key].shape
            with torch.no_grad():
                sd[target_key].copy_(sd_hf[k].t())
        else:
            assert sd_hf[k].shape == sd[target_key].shape
            with torch.no_grad():
                sd[target_key].copy_(sd_hf[k])

    print(f"Successfully loaded {model_type} weights!")
    return model


# 将方法绑定到 GPT 类
GPT.from_pretrained = from_pretrained

In [ ]:
# 加载预训练 GPT-2 权重（首次运行会下载 ~500MB）
model = GPT.from_pretrained('gpt2')
model.eval()
model.to(device)
print(f"Model on {device}")

## Part 7：简单的文本生成（Greedy Decoding）

先实现最简单的贪心解码（每次选概率最高的 token），验证模型能正常生成。

更复杂的采样策略（Temperature / Top-K / Top-P）将在 Day 4 实现。

In [ ]:
@torch.no_grad()
def generate_greedy(
    model: GPT,
    idx: torch.Tensor,
    max_new_tokens: int = 50,
) -> torch.Tensor:
    """
    贪心解码：每步选择概率最高的 token。
    
    Args:
        model: GPT 模型
        idx: 初始 token 序列, shape (B, T)
        max_new_tokens: 最多生成多少个新 token
    Returns:
        生成的完整序列, shape (B, T + max_new_tokens)
    """
    model.eval()
    for _ in range(max_new_tokens):
        # 截断到 block_size（上下文窗口限制）
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]

        # 前向传播
        logits, _ = model(idx_cond)

        # 只取最后一个位置的 logits
        logits = logits[:, -1, :]  # (B, V)

        # 贪心：选概率最高的
        idx_next = logits.argmax(dim=-1, keepdim=True)  # (B, 1)

        # 拼接到序列
        idx = torch.cat([idx, idx_next], dim=1)

    return idx

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('gpt2')

# 测试生成
prompt = "The meaning of life is"
input_ids = torch.tensor([enc.encode(prompt)], device=device)
print(f"Prompt: {prompt}")
print(f"Input IDs: {input_ids.tolist()}")
print(f"Input length: {input_ids.shape[1]}")
print("---")

output_ids = generate_greedy(model, input_ids, max_new_tokens=50)
generated_text = enc.decode(output_ids[0].tolist())
print(f"Generated: {generated_text}")

## Part 8：与 HuggingFace 输出对比验证

确认我们的实现与官方实现输出完全一致。

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# HuggingFace 官方模型
tokenizer_hf = GPT2Tokenizer.from_pretrained('gpt2')
model_hf = GPT2LMHeadModel.from_pretrained('gpt2').to(device).eval()

# 相同输入
prompt = "Alan Turing theorized that computers would one day become"
input_ids = torch.tensor([enc.encode(prompt)], device=device)

# 我们的模型
with torch.no_grad():
    our_logits, _ = model(input_ids)

# HuggingFace 模型
with torch.no_grad():
    hf_logits = model_hf(input_ids).logits

# 对比
max_diff = (our_logits - hf_logits).abs().max().item()
print(f"Max absolute difference: {max_diff:.6e}")
print(f"Match: {'✅ YES' if max_diff < 1e-4 else '❌ NO'}")

# 对比 top-5 预测
for name, logits_tensor in [("Ours", our_logits), ("HF", hf_logits)]:
    probs = F.softmax(logits_tensor[0, -1], dim=-1)
    topk = torch.topk(probs, 5)
    print(f"\n{name} Top-5 predictions for next token:")
    for i, (prob, idx) in enumerate(zip(topk.values, topk.indices)):
        print(f"  {i+1}. '{enc.decode([idx.item()])}' ({prob.item():.4f})")

## Part 9：参数量分析

验证我们的理论参数量计算。

In [ ]:
def count_parameters(model: GPT) -> dict:
    """分组统计模型参数量。"""
    counts = {}

    # Token Embedding (与 LM Head 共享，只算一次)
    counts['token_embedding'] = model.transformer.wte.weight.numel()
    counts['position_embedding'] = model.transformer.wpe.weight.numel()

    # 每层 Block
    block_total = 0
    for block in model.transformer.h:
        block_params = sum(p.numel() for p in block.parameters())
        block_total += block_params
    counts['blocks_total'] = block_total
    counts['per_block'] = block_total // model.config.n_layer

    # Final LayerNorm
    counts['final_ln'] = sum(p.numel() for p in model.transformer.ln_f.parameters())

    # 总计（权重共享后）
    counts['total'] = sum(p.numel() for p in model.parameters())

    return counts


counts = count_parameters(model)
print("=" * 50)
print("GPT-2 Small Parameter Breakdown")
print("=" * 50)
for name, count in counts.items():
    print(f"{name:25s}: {count:>12,} ({count/1e6:.1f}M)")
print("=" * 50)

## Part 10：模型内部可视化 — 注意力模式

观察预训练 GPT-2 的注意力模式，理解不同层、不同头学到了什么。

In [ ]:
def get_attention_maps(model: GPT, input_ids: torch.Tensor) -> list[torch.Tensor]:
    """获取每一层的注意力权重。临时 hook 注意力层。"""
    attention_maps = []

    def hook_fn(module, input, output):
        x = input[0]
        B, T, C = x.size()
        qkv = module.c_attn(x)
        q, k, _ = qkv.split(module.n_embd, dim=2)
        q = q.view(B, T, module.n_head, module.d_head).transpose(1, 2)
        k = k.view(B, T, module.n_head, module.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(module.d_head))
        att = att.masked_fill(module.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        attention_maps.append(att.detach().cpu())

    hooks = []
    for block in model.transformer.h:
        hooks.append(block.attn.register_forward_hook(hook_fn))

    with torch.no_grad():
        model(input_ids)

    for h in hooks:
        h.remove()

    return attention_maps


# 获取注意力图
prompt = "The cat sat on the mat"
tokens = enc.encode(prompt)
token_strs = [enc.decode([t]) for t in tokens]
input_ids = torch.tensor([tokens], device=device)

attn_maps = get_attention_maps(model, input_ids)
print(f"Got attention maps for {len(attn_maps)} layers")
print(f"Each map shape: {attn_maps[0].shape}")  # (1, n_head, T, T)

In [ ]:
# 可视化前 4 层、每层前 4 个注意力头
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for layer_idx in range(4):
    for head_idx in range(4):
        ax = axes[layer_idx][head_idx]
        att = attn_maps[layer_idx][0, head_idx].numpy()  # (T, T)
        ax.imshow(att, cmap='Blues', vmin=0, vmax=1)
        ax.set_title(f'Layer {layer_idx}, Head {head_idx}', fontsize=9)
        if layer_idx == 3:
            ax.set_xticks(range(len(token_strs)))
            ax.set_xticklabels(token_strs, rotation=45, fontsize=7)
        else:
            ax.set_xticks([])
        if head_idx == 0:
            ax.set_yticks(range(len(token_strs)))
            ax.set_yticklabels(token_strs, fontsize=7)
        else:
            ax.set_yticks([])

plt.suptitle('GPT-2 Attention Patterns (Layer × Head)', fontsize=14)
plt.tight_layout()
plt.show()

print("\n观察要点：")
print("1. 有些头学到了 'attend to previous token' 的模式")
print("2. 有些头学到了 'attend to first token' 的模式")
print("3. 深层的注意力模式比浅层更分散、更复杂")

## 本日总结

### 我们实现了什么

| 组件 | 核心要点 |
|------|----------|
| `CausalSelfAttention` | QKV 合并投影 → 分头 → 因果掩码 → softmax → 加权求和 |
| `MLP` | 两层 FFN，GELU 激活，4x 扩展 |
| `GPTBlock` | Pre-Norm + Attention + Residual + Pre-Norm + FFN + Residual |
| `GPT` | Token Emb + Pos Emb → N×Block → Final LN → LM Head（权重共享）|

### 关键细节

1. **因果掩码**：下三角矩阵，未来位置设为 $-\infty$，softmax 后变为 0
2. **Pre-Norm**：LayerNorm 在子层之前，残差连接跳过整个子层
3. **权重共享**：LM Head 与 Token Embedding 共享权重
4. **残差投影特殊初始化**：$\mathcal{N}(0, 0.02/\sqrt{2N})$

### 自检清单

- [ ] 能闭卷手写 `CausalSelfAttention`
- [ ] 能闭卷手写 `GPTBlock`
- [ ] 能闭卷手写完整 `GPT` 模型
- [ ] 理解权重加载和形状映射
- [ ] 能解释注意力可视化中的模式